# 🤖 Layoff Risk Prediction — MLOps Training Notebook

This notebook trains a **layoff risk classifier** using real-world tech layoff data scraped from LinkedIn, company news, and live sources.

**Pipeline overview:**
1. Load & merge all data sources
2. Feature engineering (industry risk scores, AI exposure, workforce size bands)
3. Train / evaluate multiple models → select best
4. Persist model + preprocessor artifacts for FastAPI inference

---

## 1. Dependencies

In [ ]:
# Core
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import xgboost as xgb

# Viz
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

# Paths
DATA_DIR   = './data'
MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

SEED = 42
print('✅ All imports OK')

## 2. Load Data Sources

In [ ]:
# Primary training table
df_layoffs = pd.read_csv(f'{DATA_DIR}/tech_layoffs_2025_2026.csv')

# Enrichment tables
df_industry  = pd.read_csv(f'{DATA_DIR}/layoffs_industry_analysis.csv')
df_dept      = pd.read_csv(f'{DATA_DIR}/layoffs_department_analysis.csv')
df_ai_impact = pd.read_csv(f'{DATA_DIR}/layoffs_ai_impact_analysis.csv')
df_hiring    = pd.read_csv(f'{DATA_DIR}/tech_hiring_trends_2025_2026.csv')
df_temporal  = pd.read_csv(f'{DATA_DIR}/layoffs_temporal_trends.csv')

print(f'Primary layoff events  : {len(df_layoffs):,} rows')
print(f'Industry analytics     : {len(df_industry):,} rows')
print(f'Department analytics   : {len(df_dept):,} rows')
print(f'AI impact breakdown    : {len(df_ai_impact):,} rows')
print(f'Hiring trends (2025-26): {len(df_hiring):,} rows')
df_layoffs.head(3)

## 3. Define Target Variable

We define **high-risk** as a layoff event that affected **> 10% of the workforce** — reflecting a structural, large-scale reduction rather than routine attrition.

In [ ]:
HIGH_RISK_THRESHOLD = 10.0  # % of total workforce laid off

df_layoffs['layoff_risk'] = (df_layoffs['Percentage_Workforce'] > HIGH_RISK_THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution of workforce % laid off
axes[0].hist(df_layoffs['Percentage_Workforce'], bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(HIGH_RISK_THRESHOLD, color='crimson', linestyle='--', linewidth=1.8, label=f'Threshold ({HIGH_RISK_THRESHOLD}%)')
axes[0].set_title('Workforce % Affected Distribution')
axes[0].set_xlabel('% of Workforce Laid Off')
axes[0].legend()

# Class balance
counts = df_layoffs['layoff_risk'].value_counts()
axes[1].bar(['Low Risk (0)', 'High Risk (1)'], counts.values, color=['#2ecc71', '#e74c3c'])
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')
axes[1].set_title('Target Class Balance')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'High risk: {counts[1]} ({counts[1]/len(df_layoffs)*100:.1f}%)')
print(f'Low risk : {counts[0]} ({counts[0]/len(df_layoffs)*100:.1f}%)')

## 4. Feature Engineering

In [ ]:
# ── 4.1  Industry-level risk score (avg workforce % laid off per industry) ──
industry_risk = (
    df_industry[['Industry', 'Avg_Workforce_Percentage']]
    .rename(columns={'Avg_Workforce_Percentage': 'industry_avg_workforce_pct'})
)

df = df_layoffs.merge(industry_risk, on='Industry', how='left')

# ── 4.2  AI exposure numeric ──
ai_map = {'No': 0, 'Partial': 1, 'Yes': 2}
df['ai_exposure_num'] = df['AI_Related'].map(ai_map)

# ── 4.3  Workforce size band (log bins) ──
df['workforce_log'] = np.log1p(df['Total_Employees'])
df['workforce_band'] = pd.qcut(
    df['Total_Employees'], q=4,
    labels=['small', 'mid', 'large', 'enterprise']
).astype(str)

# ── 4.4  Primary department (first listed) ──
df['primary_dept'] = df['Department'].str.split(',').str[0].str.strip()

# ── 4.5  Hiring signal — avg open positions per industry from hiring trends ──
hiring_signal = (
    df_hiring.groupby('Department')['Number_Positions']
    .mean()
    .reset_index()
    .rename(columns={'Department': 'primary_dept', 'Number_Positions': 'avg_open_positions'})
)
df = df.merge(hiring_signal, on='primary_dept', how='left')
df['avg_open_positions'].fillna(df['avg_open_positions'].median(), inplace=True)

# ── 4.6  Quarter risk score (Q1 layoffs tend to be higher) ──
quarter_risk = df_temporal.groupby('Quarter')['Total_Layoffs'].sum().to_dict()
max_q = max(quarter_risk.values())
df['quarter_risk_score'] = df['Quarter'].map(lambda q: quarter_risk.get(q, 0) / max_q)

# ── 4.7  Reason category (simplified) ──
def categorize_reason(reason):
    r = str(reason).lower()
    if 'ai' in r or 'automat' in r: return 'ai_automation'
    if 'profit' in r or 'cost' in r: return 'financial'
    if 'restructur' in r or 'reorg' in r: return 'restructuring'
    if 'market' in r or 'downturn' in r: return 'market_conditions'
    return 'other'

df['reason_category'] = df['Reason'].apply(categorize_reason)

print(f'Feature engineering complete. Shape: {df.shape}')
df[['Industry', 'industry_avg_workforce_pct', 'ai_exposure_num',
    'workforce_band', 'quarter_risk_score', 'reason_category']].head(5)

## 5. EDA — Key Feature Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Layoff risk by industry (top 10)
top_industries = df['Industry'].value_counts().head(10).index
industry_risk_rate = (
    df[df['Industry'].isin(top_industries)]
    .groupby('Industry')['layoff_risk'].mean()
    .sort_values(ascending=False)
)
industry_risk_rate.plot(kind='barh', ax=axes[0,0], color='#e74c3c')
axes[0,0].set_title('High-Risk Rate by Industry (Top 10)')
axes[0,0].set_xlabel('Proportion High-Risk')
axes[0,0].axvline(0.5, linestyle='--', color='gray')

# AI exposure vs risk
ai_risk = df.groupby('AI_Related')['layoff_risk'].mean().sort_values(ascending=False)
colors = ['#e74c3c', '#f39c12', '#2ecc71']
ai_risk.plot(kind='bar', ax=axes[0,1], color=colors, rot=0)
axes[0,1].set_title('High-Risk Rate by AI Exposure')
axes[0,1].set_ylabel('Proportion High-Risk')
axes[0,1].set_xlabel('')

# Workforce size vs % laid off
axes[1,0].scatter(
    df['workforce_log'], df['Percentage_Workforce'],
    c=df['layoff_risk'], cmap='RdYlGn_r', alpha=0.8, edgecolors='white', s=80
)
axes[1,0].axhline(HIGH_RISK_THRESHOLD, color='crimson', linestyle='--', linewidth=1.5)
axes[1,0].set_title('Log Workforce Size vs % Laid Off')
axes[1,0].set_xlabel('Log(Total Employees)')
axes[1,0].set_ylabel('% Workforce Laid Off')

# Reason category breakdown
reason_risk = df.groupby('reason_category')['layoff_risk'].mean().sort_values(ascending=True)
reason_risk.plot(kind='barh', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('High-Risk Rate by Layoff Reason')
axes[1,1].set_xlabel('Proportion High-Risk')

plt.suptitle('Exploratory Analysis — Layoff Risk Factors', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Prepare Training Data

In [ ]:
# ── Feature selection ──
NUMERIC_FEATURES = [
    'Employees_Laid_Off',
    'Severance_Weeks',
    'Total_Employees',
    'workforce_log',
    'ai_exposure_num',
    'industry_avg_workforce_pct',
    'avg_open_positions',
    'quarter_risk_score',
    'Month',
    'Quarter',
]

CATEGORICAL_FEATURES = [
    'Industry',
    'reason_category',
    'workforce_band',
    'primary_dept',
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET       = 'layoff_risk'

X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

# Encode categoricals for inspection
print('Features summary:')
print(f'  Numeric      : {NUMERIC_FEATURES}')
print(f'  Categorical  : {CATEGORICAL_FEATURES}')
print(f'  Total samples: {len(X)}')
X.info()

In [ ]:
# ── Train / test split (stratified) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

# ── Preprocessing pipeline ──
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,  NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

print('✅ Preprocessor defined')

## 7. Train & Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEED),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=SEED),
    'XGBoost':             xgb.XGBClassifier(
                               n_estimators=200, learning_rate=0.05, max_depth=4,
                               use_label_encoder=False, eval_metric='logloss',
                               random_state=SEED, verbosity=0
                           ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    results[name] = {
        'mean_auc': scores.mean(),
        'std_auc':  scores.std(),
        'pipeline': pipe,
    }
    print(f'{name:<25} CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# ── Visualize CV scores ──
names   = list(results.keys())
means   = [results[n]['mean_auc'] for n in names]
stds    = [results[n]['std_auc']  for n in names]
colors  = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, means, xerr=stds, color=colors, capsize=5, edgecolor='white')
ax.set_xlim(0.4, 1.0)
ax.axvline(0.8, linestyle='--', color='gray', alpha=0.6)
for bar, mean in zip(bars, means):
    ax.text(mean + 0.01, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_title('5-Fold CV ROC-AUC by Model', fontsize=13, fontweight='bold')
ax.set_xlabel('ROC-AUC')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_name = max(results, key=lambda n: results[n]['mean_auc'])
print(f'\n🏆 Best model: {best_name} (AUC={results[best_name]["mean_auc"]:.4f})')

## 8. Final Model — Train on Full Train Set & Evaluate on Test Set

In [ ]:
best_pipeline = results[best_name]['pipeline']
best_pipeline.fit(X_train, y_train)

y_pred      = best_pipeline.predict(X_test)
y_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]
test_auc    = roc_auc_score(y_test, y_pred_prob)

print(f'Test ROC-AUC : {test_auc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Low Risk', 'High Risk'],
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=axes[1], name=best_name)
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4)
axes[1].set_title(f'ROC Curve — {best_name}')

plt.suptitle(f'Test Set Evaluation | AUC={test_auc:.4f}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Importance

In [ ]:
clf = best_pipeline.named_steps['classifier']

if hasattr(clf, 'feature_importances_'):
    feature_names = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    importances   = clf.feature_importances_
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    colors = ['#e74c3c' if f in CATEGORICAL_FEATURES else '#3498db' for f in fi_df['feature']]
    ax.barh(fi_df['feature'], fi_df['importance'], color=colors)
    ax.set_title(f'Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')

    from matplotlib.patches import Patch
    legend = [Patch(color='#3498db', label='Numeric'), Patch(color='#e74c3c', label='Categorical')]
    ax.legend(handles=legend, loc='lower right')

    plt.tight_layout()
    plt.savefig(f'{MODELS_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Feature importance not available for this model type.')

## 10. Save Model Artifacts for FastAPI

In [ ]:
# Retrain on full dataset for production
best_pipeline.fit(X, y)

# Save pipeline
model_path = f'{MODELS_DIR}/layoff_risk_model.pkl'
joblib.dump(best_pipeline, model_path)
print(f'✅ Model saved → {model_path}')

# Save feature schema (needed by FastAPI for request validation)
schema = {
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'all_features': ALL_FEATURES,
    'target': TARGET,
    'risk_threshold': HIGH_RISK_THRESHOLD,
    'best_model': best_name,
    'test_auc': round(test_auc, 4),
    'industries': sorted(df['Industry'].unique().tolist()),
    'departments': sorted(df['primary_dept'].unique().tolist()),
    'reason_categories': sorted(df['reason_category'].unique().tolist()),
    'ai_exposure_map': ai_map,
    'industry_avg_pct_map': df.groupby('Industry')['industry_avg_workforce_pct'].first().to_dict(),
    'median_open_positions': float(df['avg_open_positions'].median()),
    'quarter_risk_map': {str(k): float(v) for k, v in 
                         df.set_index('Quarter')['quarter_risk_score'].to_dict().items()},
}

schema_path = f'{MODELS_DIR}/model_schema.json'
with open(schema_path, 'w') as f:
    json.dump(schema, f, indent=2)
print(f'✅ Schema saved → {schema_path}')

# Quick smoke test
sample = X.iloc[[0]]
prob   = best_pipeline.predict_proba(sample)[0][1]
print(f'\n🔬 Smoke test — sample risk probability: {prob:.4f}')

## 11. Continuous Learning Hook

Since new layoff data is ingested from live sources (LinkedIn, news feeds), this cell re-trains on fresh data dropped into the `data/` directory.
Trigger it via a scheduled Airflow/cron job after each data refresh.

In [ ]:
def retrain_on_new_data(new_data_path: str, model_path: str = model_path, schema_path: str = schema_path):
    """
    Incrementally retrain the layoff risk model with new data.
    
    Args:
        new_data_path: Path to a CSV with the same schema as tech_layoffs_2025_2026.csv
        model_path:    Path to existing pickled pipeline
        schema_path:   Path to model_schema.json
    
    The function:
    1. Loads existing + new data
    2. Runs the same feature engineering pipeline
    3. Retrains from scratch (XGBoost / RF models benefit more from full retrain than warm-start)
    4. Overwrites the artifact if new AUC >= old AUC (champion-challenger)
    """
    import json, joblib

    with open(schema_path) as f:
        schema = json.load(f)

    existing_model = joblib.load(model_path)
    old_auc        = schema['test_auc']

    # Load & concatenate
    df_old = pd.read_csv(f'{DATA_DIR}/tech_layoffs_2025_2026.csv')
    df_new = pd.read_csv(new_data_path)
    df_combined = pd.concat([df_old, df_new], ignore_index=True).drop_duplicates()

    print(f'Combined dataset: {len(df_combined)} rows (+{len(df_new)} new)')

    # Feature engineering (same steps as above)
    df_combined['layoff_risk']        = (df_combined['Percentage_Workforce'] > HIGH_RISK_THRESHOLD).astype(int)
    df_combined['ai_exposure_num']    = df_combined['AI_Related'].map(ai_map)
    df_combined['workforce_log']      = np.log1p(df_combined['Total_Employees'])
    df_combined['workforce_band']     = pd.qcut(df_combined['Total_Employees'], q=4,
                                                labels=['small','mid','large','enterprise']).astype(str)
    df_combined['primary_dept']       = df_combined['Department'].str.split(',').str[0].str.strip()
    df_combined['reason_category']    = df_combined['Reason'].apply(categorize_reason)
    df_combined = df_combined.merge(industry_risk, on='Industry', how='left')
    df_combined = df_combined.merge(hiring_signal, on='primary_dept', how='left')
    df_combined['avg_open_positions'].fillna(df_combined['avg_open_positions'].median(), inplace=True)
    df_combined['quarter_risk_score'] = df_combined['Quarter'].map(lambda q: quarter_risk.get(q, 0) / max_q)

    X_new = df_combined[ALL_FEATURES]
    y_new = df_combined[TARGET]

    X_tr, X_te, y_tr, y_te = train_test_split(X_new, y_new, test_size=0.2, random_state=SEED, stratify=y_new)

    # Build fresh pipeline with same best model type
    new_clf = type(clf)(**clf.get_params()) if hasattr(clf, 'get_params') else clf
    new_pipe = Pipeline([('preprocessor', preprocessor), ('classifier', new_clf)])
    new_pipe.fit(X_tr, y_tr)

    new_auc = roc_auc_score(y_te, new_pipe.predict_proba(X_te)[:, 1])
    print(f'Old AUC: {old_auc:.4f} | New AUC: {new_auc:.4f}')

    if new_auc >= old_auc:
        new_pipe.fit(X_new, y_new)  # retrain on full combined set
        joblib.dump(new_pipe, model_path)
        schema['test_auc'] = round(new_auc, 4)
        with open(schema_path, 'w') as f:
            json.dump(schema, f, indent=2)
        print(f'✅ Champion updated! Model saved with AUC={new_auc:.4f}')
    else:
        print('⚠️  New model did not beat champion. Keeping existing model.')

print('Continuous learning hook ready.')
print('Call retrain_on_new_data("data/new_layoffs.csv") after each data refresh.')

---
## ✅ Summary

| Step | Output |
|------|--------|
| Data | `tech_layoffs_2025_2026.csv` + 5 enrichment tables |
| Features | 10 numeric + 4 categorical |
| Models compared | Logistic Regression, Random Forest, GBM, XGBoost |
| Artifacts | `models/layoff_risk_model.pkl`, `models/model_schema.json` |
| Next step | Run `uvicorn app:app --reload` to serve predictions |
